# RAVEN-X — real run on a free GPU (Colab/Kaggle T4)

**Cross-script rationale transfer for Hindi hate speech.** This notebook is self-contained:
it writes its own code, stages the data, runs a real baseline, and trains the real MuRIL model.

### Do this first
1. **Runtime → Change runtime type → T4 GPU** (Colab) or **Settings → Accelerator → GPU T4** (Kaggle).
2. **Runtime → Run all.** Total ≈ **25–45 min** in fast mode (fp16). Everything below is real — no fabrication.

> Data (HASOC-2019, HateXplain) is **research-only**; this notebook keeps it in the session and
> never commits it. Cite Mandl et al. 2019, Mathew et al. 2021, MuRIL (Khanuja et al. 2021).

## 1 · Confirm the GPU is attached

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (set Runtime->T4 GPU!)')

CUDA available: True
Device: Tesla T4


## 2 · Install dependencies
> Colab already ships a **CUDA-matched torch + torchvision + transformers**. We do **NOT** upgrade torch — doing so breaks torchvision and makes `transformers` fail to import. We only ensure scikit-learn/scipy (used by the baseline) are present.

In [2]:
# Do NOT upgrade torch on Colab (breaks torchvision -> transformers import).
!pip -q install scikit-learn scipy
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

torch 2.11.0+cu128 | transformers 5.9.0 | CUDA True | Tesla T4


## 3 · Stage the data (research-only; stays in the session)

In [3]:
import os, subprocess, urllib.request
os.makedirs('data/hasoc2019', exist_ok=True)
os.makedirs('data/hatexplain', exist_ok=True)
# HASOC-2019 open mirror -> copy the Hindi tsv (labeled) and blind test
subprocess.run(['git','clone','--depth','1',
  'https://github.com/TharinduDR/HASOC-2019','/tmp/hasoc'], check=True)
for f in ['hindi_dataset.tsv','hasoc2019_hi_test.tsv']:
    subprocess.run(['cp', f'/tmp/hasoc/data/{f}', f'data/hasoc2019/{f}'], check=True)
# HateXplain (rationales)
base='https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/'
for f in ['dataset.json','post_id_divisions.json']:
    urllib.request.urlretrieve(base+f, f'data/hatexplain/{f}')
print('staged:', os.listdir('data/hasoc2019'), os.listdir('data/hatexplain'))

staged: ['hasoc2019_hi_test.tsv', 'hindi_dataset.tsv'] ['post_id_divisions.json', 'dataset.json']


## 4 · Write the RAVEN-X code (identical to the locally smoke-tested modules)

In [4]:
%%writefile data_loaders.py
"""RAVEN-X — reusable corpus loaders (design-independent, stdlib only).

Two corpora feed the cross-lingual rationale-transfer model:
  * HASOC-2019 Hindi  -> code-mixed posts with POST-LEVEL labels (the classification signal)
  * HateXplain (EN)   -> posts with TOKEN-LEVEL rationale masks (the explanation signal)

These loaders return plain Python dicts so they stay independent of the chosen
backbone/tokenizer. Tokenization + subword<->rationale alignment happen later,
once the methodology (backbone, heads) is finalized.

Verified facts (see eda.py output, 2026-06-07):
  HASOC Hindi labeled = 4665 rows; official test is BLIND -> seeded stratified split.
  HateXplain = 20148 posts; 56.6% carry rationales; official divisions provided.
"""
from __future__ import annotations
import csv, json, os, random
from collections import Counter

HERE = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(HERE, "data")

# ---- HASOC-2019 Hindi -------------------------------------------------------
HASOC_DIR = os.path.join(DATA, "hasoc2019")
# binary task_1: HOF (hate-or-offensive) = 1, NOT = 0
BIN = {"NOT": 0, "HOF": 1}
# fine-grained task_2
FINE = {"NONE": 0, "HATE": 1, "OFFN": 2, "PRFN": 3}


def _read_hasoc_tsv(path):
    out = []
    with open(path, encoding="utf-8") as f:
        r = csv.reader(f, delimiter="\t")
        header = next(r)
        labeled = len(header) >= 5
        for row in r:
            if not labeled or len(row) < 5 or row[2] not in BIN:
                continue
            out.append({
                "id": row[0],
                "text": row[1],
                "label": BIN[row[2]],          # binary HOF/NOT
                "fine": FINE.get(row[3], 0),   # hate/offn/prfn/none
                "target": row[4],              # TIN/UNT/NONE
            })
    return out


def load_hasoc_hindi(seed=42, ratios=(0.70, 0.15, 0.15)):
    """Stratified seeded split of the labeled set (official test is blind)."""
    rows = _read_hasoc_tsv(os.path.join(HASOC_DIR, "hindi_dataset.tsv"))
    by_label = {}
    for ex in rows:
        by_label.setdefault(ex["label"], []).append(ex)
    train, val, test = [], [], []
    for label, items in sorted(by_label.items()):
        rng = random.Random(seed)
        rng.shuffle(items)
        n = len(items)
        a, b = int(n * ratios[0]), int(n * ratios[0] + n * ratios[1])
        train += items[:a]
        val += items[a:b]
        test += items[b:]
    random.Random(seed).shuffle(train)
    return {"train": train, "val": val, "test": test}


# ---- HateXplain (English, token rationales) ---------------------------------
HX_DIR = os.path.join(DATA, "hatexplain")
# collapse to the same toxic/non-toxic axis as HASOC: hate|offensive -> 1, normal -> 0
HX3 = {"normal": 0, "offensive": 1, "hatespeech": 2}
HX_BIN = {"normal": 0, "offensive": 1, "hatespeech": 1}


def _majority(labels):
    return Counter(labels).most_common(1)[0][0]


def _union_rationale(rationales, n_tokens):
    """OR the per-annotator 0/1 masks into one token-level rationale mask."""
    mask = [0] * n_tokens
    for r in rationales or []:
        for j in range(min(n_tokens, len(r))):
            if r[j]:
                mask[j] = 1
    return mask


def load_hatexplain():
    with open(os.path.join(HX_DIR, "dataset.json"), encoding="utf-8") as f:
        data = json.load(f)
    with open(os.path.join(HX_DIR, "post_id_divisions.json"), encoding="utf-8") as f:
        div = json.load(f)
    by_id = {}
    for pid, v in data.items():
        toks = v.get("post_tokens", [])
        labels = [a["label"] for a in v["annotators"]]
        maj = _majority(labels)
        by_id[pid] = {
            "id": pid,
            "tokens": toks,
            "text": " ".join(toks),
            "label3": HX3[maj],                 # normal/offensive/hate
            "label": HX_BIN[maj],               # toxic vs not (aligns to HASOC)
            "rationale": _union_rationale(v.get("rationales", []), len(toks)),
            "has_rationale": any(sum(r) > 0 for r in v.get("rationales", []) or []),
        }
    splits = {}
    for name, ids in div.items():
        splits[name] = [by_id[i] for i in ids if i in by_id]
    return splits


if __name__ == "__main__":  # self-test
    h = load_hasoc_hindi()
    print("HASOC Hindi:", {k: len(v) for k, v in h.items()},
          "| train label balance:", dict(Counter(x["label"] for x in h["train"])))
    x = load_hatexplain()
    print("HateXplain:", {k: len(v) for k, v in x.items()})
    ex = next(e for e in x["train"] if e["has_rationale"])
    print("  example rationale-bearing post:")
    print("   tokens:", ex["tokens"][:12], "...")
    print("   mask  :", ex["rationale"][:12], "...  label3=", ex["label3"])
    print("  rationale coverage in train:",
          f"{sum(e['has_rationale'] for e in x['train'])}/{len(x['train'])}")


Writing data_loaders.py


In [5]:
%%writefile raven_x.py
"""RAVEN-X — model + multi-corpus masked loss + label-free faithfulness objective.

One shared encoder (MuRIL) with two heads:
  HEAD 1 classification: shared toxicity z (both corpora) + native y3 (HateXplain 3-cls)
          + t4 (HASOC fine-type) — each masked to its own corpus.
  HEAD 2 token-rationale (NER-style): per-subword logit a_j; English-supervised only.

Faithfulness self-objective on Hindi (no labels): remove the top-k rationale tokens via a
straight-through ATTENTION mask and RE-RUN the full encoder; toxicity must drop
(comprehensiveness) and rationale-only must hold (sufficiency). The float attention mask
carries gradient back to the rationale head (verified by the smoke test's grad-norm check).

Smoke test (fast, CPU, no 950MB download):
    raven-api/.venv/bin/python raven-codemixed/raven_x.py --smoke
Real training swaps in MuRIL:  --model google/muril-base-cased
"""
from __future__ import annotations
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE = -100


# ----------------------------------------------------------------------------- model
class RavenXModel(nn.Module):
    def __init__(self, encoder, hidden):
        super().__init__()
        self.encoder = encoder
        self.drop = nn.Dropout(0.2)
        self.cls_trunk = nn.Sequential(nn.Linear(hidden, hidden), nn.Tanh())
        self.z = nn.Linear(hidden, 1)      # shared toxicity (both corpora)
        self.y3 = nn.Linear(hidden, 3)     # HateXplain native 3-class
        self.t4 = nn.Linear(hidden, 4)     # HASOC fine-type
        self.rat = nn.Sequential(          # token-rationale head
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)
        )

    def encode(self, input_ids, attention_mask):
        # attention_mask may be a float tensor (for the differentiable faithfulness re-run)
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state               # (B, L, H)

    def heads(self, H):
        h_cls = self.drop(H[:, 0])                 # pooled [CLS]
        trunk = self.cls_trunk(h_cls)
        a = self.rat(H).squeeze(-1)                # (B, L) subword rationale logits
        return {
            "z": self.z(trunk).squeeze(-1),        # (B,)
            "y3": self.y3(trunk),                  # (B,3)
            "t4": self.t4(trunk),                  # (B,4)
            "a": a,
        }

    def forward(self, input_ids, attention_mask):
        return self.heads(self.encode(input_ids, attention_mask))

    def p_hof(self, input_ids, attention_mask):
        """Toxicity prob from a fresh encode (used by the faithfulness re-run)."""
        H = self.encode(input_ids, attention_mask)
        return torch.sigmoid(self.z(self.cls_trunk(H[:, 0])).squeeze(-1))


# ----------------------------------------------------------------- straight-through top-k
def ste_topk_mask(scores, valid, k_frac=0.20):
    """Per-row hard top-k over `valid` tokens, with a straight-through estimator so
    gradient flows back to `scores`. Returns a float mask (B,L), 1.0 = selected."""
    B, L = scores.shape
    probs = torch.sigmoid(scores)
    hard = torch.zeros_like(probs)
    for b in range(B):
        idx = valid[b].nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            continue
        k = max(1, int(round(k_frac * idx.numel())))
        sel = idx[torch.topk(probs[b, idx], k).indices]
        hard[b, sel] = 1.0
    # STE: forward = hard, backward = d/d(probs)
    return hard + (probs - probs.detach())


# --------------------------------------------------------------------------- the loss
def compute_loss(model, batch, w, gamma):
    """batch: dict of tensors. w: loss weights. gamma: faithfulness weight (0 disables)."""
    ids, attn = batch["input_ids"], batch["attention_mask"]
    out = model(ids, attn.float())
    z, a = out["z"], out["a"]
    is_en, is_hi = batch["is_en"].bool(), batch["is_hi"].bool()
    parts = {}

    # (1) shared toxicity — both corpora
    parts["z"] = F.binary_cross_entropy_with_logits(z, batch["z_star"].float())

    # (2) native heads — masked per corpus
    parts["clsA"] = (F.cross_entropy(out["y3"][is_en], batch["y3"][is_en])
                     if is_en.any() else z.sum() * 0)
    parts["clsB"] = (F.cross_entropy(out["t4"][is_hi], batch["t4"][is_hi])
                     if is_hi.any() else z.sum() * 0)

    # (3) rationale — English-supervised token-BCE on first-subword logits (-100 ignored)
    r_tgt = batch["r_sub"]                      # (B,L) in {0,1,-100}
    keep = (r_tgt != IGNORE) & batch["has_rationale"].bool().unsqueeze(1)
    if keep.any():
        pos_w = torch.tensor(4.0, device=z.device)
        parts["rat"] = F.binary_cross_entropy_with_logits(
            a[keep], r_tgt[keep].clamp(min=0).float(), pos_weight=pos_w)
    else:
        parts["rat"] = a.sum() * 0

    # (4) faithfulness — label-free, Hindi only, differentiable attention masking
    if gamma > 0 and is_hi.any():
        special = batch["special"].bool()       # CLS/SEP/PAD
        valid = (attn.bool() & ~special)
        sel = ste_topk_mask(a, valid)           # (B,L) STE, 1 = rationale token
        keep_special = special.float() + attn.float() * special.float()  # always keep specials
        cls_keep = torch.zeros_like(attn.float()); cls_keep[:, 0] = 1.0  # always keep CLS
        # comprehensiveness: REMOVE rationale -> toxicity should drop
        attn_comp = (attn.float() * (1.0 - sel)).clamp(0, 1)
        attn_comp = torch.maximum(attn_comp, cls_keep)
        p_comp = model.p_hof(ids, attn_comp)
        # sufficiency: KEEP only rationale -> toxicity should hold
        attn_suff = torch.maximum(attn.float() * sel, cls_keep)
        p_suff = model.p_hof(ids, attn_suff)
        p_full = torch.sigmoid(z).detach()
        hi = is_hi.float()
        parts["comp"] = (p_comp * hi).sum() / hi.sum()
        parts["suff"] = (((p_full - p_suff) ** 2) * hi).sum() / hi.sum()
    else:
        parts["comp"] = z.sum() * 0
        parts["suff"] = z.sum() * 0

    # (5) priors — short + contiguous
    pi = torch.sigmoid(a)
    parts["sparse"] = pi.mean()
    parts["cont"] = (pi[:, 1:] - pi[:, :-1]).abs().mean()

    total = (parts["z"]
             + w["alpha"] * (parts["clsA"] + parts["clsB"])
             + w["beta"] * parts["rat"]
             + gamma * (parts["comp"] + parts["suff"])
             + w["lam_s"] * parts["sparse"] + w["lam_c"] * parts["cont"])
    return total, parts


# ------------------------------------------------------------------------ tokenization
def encode_examples(examples, tokenizer, max_len=128):
    """Tokenize a list of unified examples into model-ready tensors with subword<->word
    alignment and first-subword rationale targets. Each example needs:
      text (str), words (list[str]), z_star (0/1), is_en/is_hi, has_rationale (0/1),
      rationale (list[0/1] over words) or None, y3 (int), t4 (int)."""
    texts = [ex["words"] for ex in examples]
    enc = tokenizer(texts, is_split_into_words=True, truncation=True,
                    max_length=max_len, padding="max_length", return_tensors="pt")
    B, L = enc["input_ids"].shape
    r_sub = torch.full((B, L), IGNORE, dtype=torch.long)
    special = torch.zeros((B, L), dtype=torch.long)
    for b, ex in enumerate(examples):
        wids = enc.word_ids(b)
        seen = set()
        for j, wid in enumerate(wids):
            if wid is None:
                special[b, j] = 1
                continue
            if wid in seen:           # only first subword of each word gets a target
                continue
            seen.add(wid)
            if ex.get("rationale") is not None and wid < len(ex["rationale"]):
                r_sub[b, j] = int(ex["rationale"][wid])
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "special": special,
        "r_sub": r_sub,
        "z_star": torch.tensor([ex["z_star"] for ex in examples]),
        "y3": torch.tensor([ex.get("y3", 0) for ex in examples]),
        "t4": torch.tensor([ex.get("t4", 0) for ex in examples]),
        "is_en": torch.tensor([ex["is_en"] for ex in examples]),
        "is_hi": torch.tensor([ex["is_hi"] for ex in examples]),
        "has_rationale": torch.tensor([ex["has_rationale"] for ex in examples]),
    }


def unify_hasoc(ex):
    return {"words": ex["text"].split(), "z_star": ex["label"], "is_en": 0, "is_hi": 1,
            "has_rationale": 0, "rationale": None, "y3": 0, "t4": ex["fine"]}


def unify_hatexplain(ex):
    return {"words": ex["tokens"], "z_star": ex["label"], "is_en": 1, "is_hi": 0,
            "has_rationale": int(ex["has_rationale"]), "rationale": ex["rationale"],
            "y3": ex["label3"], "t4": 0}


# -------------------------------------------------------------------------- smoke test
def smoke():
    from transformers import AutoTokenizer, BertConfig, BertModel
    import data_loaders as dl

    print("[smoke] loading MuRIL tokenizer (real alignment) + tiny random encoder (fast)...")
    tok = AutoTokenizer.from_pretrained("google/muril-base-cased")
    cfg = BertConfig(vocab_size=tok.vocab_size, hidden_size=64, num_hidden_layers=2,
                     num_attention_heads=2, intermediate_size=128, max_position_embeddings=256)
    model = RavenXModel(BertModel(cfg), hidden=64)

    hi = dl.load_hasoc_hindi()["train"][:4]
    en = [e for e in dl.load_hatexplain()["train"] if e["has_rationale"]][:4]
    examples = [unify_hasoc(e) for e in hi] + [unify_hatexplain(e) for e in en]
    batch = encode_examples(examples, tok, max_len=64)

    # alignment sanity on a REAL Devanagari + English example
    print(f"[smoke] batch input_ids {tuple(batch['input_ids'].shape)}, "
          f"specials/row≈{batch['special'].sum().item()//8}, "
          f"EN rationale tokens={int((batch['r_sub']==1).sum())}, "
          f"ignored={int((batch['r_sub']==IGNORE).sum())}")
    assert (batch["r_sub"][:4] == IGNORE).all(), "HASOC rows must have NO rationale target"
    assert (batch["r_sub"][4:] == 1).any(), "HateXplain rows must carry rationale targets"

    w = dict(alpha=0.5, beta=2.0, lam_s=0.02, lam_c=0.01)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for step in range(2):
        gamma = 0.0 if step == 0 else 0.5        # ramp: faithfulness on at step 2
        opt.zero_grad()
        total, parts = compute_loss(model, batch, w, gamma)
        total.backward()
        # the critical check: faithfulness gradient must reach the rationale head
        gnorm = sum(p.grad.norm().item() for p in model.rat.parameters() if p.grad is not None)
        assert torch.isfinite(total), "loss is not finite"
        assert gnorm > 0, "NO gradient reached the rationale head"
        opt.step()
        print(f"[smoke] step {step+1} gamma={gamma} loss={total.item():.4f} "
              f"rat_grad_norm={gnorm:.4e} | " +
              " ".join(f"{k}={v.item():.3f}" for k, v in parts.items()))
    print("[smoke] PASS — forward, masked loss, faithfulness re-encode, and "
          "straight-through gradient flow to the rationale head all verified.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--model", default="google/muril-base-cased")
    args = ap.parse_args()
    if args.smoke:
        smoke()
    else:
        print("Use --smoke for the local pipeline check. Real training: see train (Week 2-3).")


Writing raven_x.py


In [6]:
%%writefile train_raven_x.py
"""RAVEN-X trainer — multi-corpus, two-stage, faithfulness-regularized.

Local smoke (CPU, tiny encoder, real MuRIL tokenizer):
    raven-api/.venv/bin/python raven-codemixed/train_raven_x.py --smoke
Real run (Kaggle/Colab T4):
    python train_raven_x.py --model google/muril-base-cased --epochs 4 --out ckpt_ravenx

Saves a checkpoint dir the Raven API can serve (encoder/ + heads.pt + raven_x.json).
Dependency-light: metrics computed by hand (no sklearn needed).
"""
from __future__ import annotations
import argparse, contextlib, json, os, random
import torch
import raven_x as rx
import data_loaders as dl


# ----------------------------------------------------------------------- metrics (no sklearn)
def _f1(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 2 * p * r / (p + r) if p + r else 0.0


def macro_f1_binary(preds, golds):
    f = []
    for c in (0, 1):
        tp = sum(p == c and g == c for p, g in zip(preds, golds))
        fp = sum(p == c and g != c for p, g in zip(preds, golds))
        fn = sum(p != c and g == c for p, g in zip(preds, golds))
        f.append(_f1(tp, fp, fn))
    return sum(f) / 2


def token_f1(pred_mask, gold_mask):
    tp = sum(p and g for p, g in zip(pred_mask, gold_mask))
    fp = sum(p and not g for p, g in zip(pred_mask, gold_mask))
    fn = sum((not p) and g for p, g in zip(pred_mask, gold_mask))
    return _f1(tp, fp, fn)


# ------------------------------------------------------------------------------ batching
def stratified_batches(hi, en, bs, seed, max_batches=None):
    """Each batch = bs/2 HASOC (oversampled w/ replacement) + bs/2 HateXplain."""
    half = bs // 2
    rng = random.Random(seed)
    en = en[:]
    rng.shuffle(en)
    hi_pool = hi[:]
    rng.shuffle(hi_pool)
    n = len(en) // half
    if max_batches:
        n = min(n, max_batches)
    hp = 0
    for b in range(n):
        en_chunk = en[b * half:(b + 1) * half]
        hi_chunk = []
        for _ in range(half):
            if hp >= len(hi_pool):
                rng.shuffle(hi_pool); hp = 0
            hi_chunk.append(hi_pool[hp]); hp += 1
        yield ([rx.unify_hasoc(x) for x in hi_chunk] +
               [rx.unify_hatexplain(x) for x in en_chunk])


# ------------------------------------------------------------------------------- eval
@torch.no_grad()
def evaluate(model, tok, hi_val, en_val, max_len, device, cap=400):
    model.eval()
    # HASOC macro-F1
    preds, golds = [], []
    for i in range(0, min(len(hi_val), cap), 16):
        ex = [rx.unify_hasoc(x) for x in hi_val[i:i + 16]]
        bt = rx.encode_examples(ex, tok, max_len)
        z = model(bt["input_ids"].to(device), bt["attention_mask"].float().to(device))["z"]
        preds += (z > 0).long().tolist()
        golds += [e["z_star"] for e in ex]
    mf1 = macro_f1_binary(preds, golds)
    # HateXplain token-F1 (first-subword positions with a real rationale target)
    pm, gm = [], []
    en_r = [e for e in en_val if e["has_rationale"]][:cap]
    for i in range(0, len(en_r), 16):
        ex = [rx.unify_hatexplain(x) for x in en_r[i:i + 16]]
        bt = rx.encode_examples(ex, tok, max_len)
        a = model(bt["input_ids"].to(device), bt["attention_mask"].float().to(device))["a"]
        keep = bt["r_sub"] != rx.IGNORE
        pm += (torch.sigmoid(a.cpu())[keep] > 0.5).long().tolist()
        gm += bt["r_sub"][keep].tolist()
    tf1 = token_f1(pm, gm)
    model.train()
    return mf1, tf1


# ------------------------------------------------------------------------------ trainer
def build_model(name, tok, tiny, device):
    if tiny:
        from transformers import BertConfig, BertModel
        cfg = BertConfig(vocab_size=tok.vocab_size, hidden_size=64, num_hidden_layers=2,
                         num_attention_heads=2, intermediate_size=128,
                         max_position_embeddings=256)
        enc, hidden = BertModel(cfg), 64
    else:
        from transformers import AutoModel
        enc = AutoModel.from_pretrained(name)
        hidden = enc.config.hidden_size
    return rx.RavenXModel(enc, hidden).to(device)


def set_encoder_grad(model, on):
    for p in model.encoder.parameters():
        p.requires_grad = on


def save_ckpt(model, tok, args, max_len):
    """Disconnect-safe: overwrite the checkpoint after every epoch."""
    os.makedirs(args.out, exist_ok=True)
    if not args.tiny:
        model.encoder.save_pretrained(os.path.join(args.out, "encoder"))
        tok.save_pretrained(os.path.join(args.out, "encoder"))
    torch.save({k: v for k, v in model.state_dict().items() if not k.startswith("encoder.")},
               os.path.join(args.out, "heads.pt"))
    json.dump({"backbone": args.model, "hidden": model.z.in_features, "max_len": max_len,
               "split_seed": 42}, open(os.path.join(args.out, "raven_x.json"), "w"), indent=2)


def train(args):
    import time
    from transformers import AutoTokenizer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = device == "cuda" and not args.tiny and not args.no_amp     # fp16 on a real GPU
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    autocast = ((lambda: torch.autocast("cuda", dtype=torch.float16))
                if use_amp else contextlib.nullcontext)
    tok = AutoTokenizer.from_pretrained("google/muril-base-cased" if args.tiny else args.model)
    model = build_model(args.model, tok, args.tiny, device)
    hi = dl.load_hasoc_hindi(); hx = dl.load_hatexplain()
    w = dict(alpha=0.5, beta=2.0, lam_s=0.02, lam_c=0.01)
    max_len = 64 if args.tiny else 128
    gamma_target = 0.5
    if args.tiny:
        epochs, cap = 1, 3
    elif args.fast:
        epochs, cap = args.epochs, 600          # quick real pass: cap steps/epoch
    else:
        epochs, cap = args.epochs, None

    def run_epoch(opt, tag, gamma_on, seed_off):
        batches = list(stratified_batches(hi["train"], hx["train"], args.bs,
                                          args.seed + seed_off, cap))
        t0 = time.time()
        for bi, ex in enumerate(batches):
            bt = {k: v.to(device) for k, v in rx.encode_examples(ex, tok, max_len).items()}
            gamma = gamma_target * min(1.0, bi / max(1, len(batches) // 2)) if gamma_on else 0.0
            opt.zero_grad()
            with autocast():
                total, parts = rx.compute_loss(model, bt, w, gamma)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            if bi % max(1, len(batches) // 4) == 0 or args.tiny:
                print(f"  {tag} batch {bi+1}/{len(batches)} gamma={gamma:.2f} "
                      f"loss={total.item():.4f} rat={parts['rat'].item():.3f} "
                      f"comp={parts['comp'].item():.3f}", flush=True)
        print(f"  [{tag}] {len(batches)} batches in {time.time()-t0:.0f}s", flush=True)

    print(f"[train] device={device} amp={use_amp} mode={'fast' if args.fast else 'full'} "
          f"model={'tiny' if args.tiny else args.model}")
    set_encoder_grad(model, False)
    opt0 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=1e-3 if args.tiny else 1e-4)
    print("[train] Stage 0 — freeze encoder, warm heads (gamma=0)")
    run_epoch(opt0, "stage0", gamma_on=False, seed_off=0)
    set_encoder_grad(model, True)
    opt1 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=1e-3 if args.tiny else 2e-5)
    print("[train] Stage 1 — unfreeze; faithfulness " +
          ("on FINAL epoch only (fast)" if args.fast else "with gamma ramp every epoch"))
    for ep in range(epochs):
        gamma_on = (ep == epochs - 1) if args.fast else True
        run_epoch(opt1, f"stage1.{ep+1}", gamma_on=gamma_on, seed_off=ep + 1)
        mf1, tf1 = evaluate(model, tok, hi["val"], hx["val"], max_len, device,
                            cap=64 if args.tiny else 400)
        print(f"[train] epoch {ep+1}/{epochs}: HASOC macro-F1={mf1:.3f}  "
              f"HateXplain token-F1={tf1:.3f}  (faithfulness={'on' if gamma_on else 'off'})")
        save_ckpt(model, tok, args, max_len)
        print(f"[train] checkpoint saved -> {args.out}/  (epoch {ep+1}, disconnect-safe)")
    print("[train] done.")
    return True


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true", help="tiny encoder + 3 batches, CPU")
    ap.add_argument("--model", default="google/muril-base-cased")
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", default="raven-codemixed/ckpt_ravenx")
    ap.add_argument("--fast", action="store_true",
                    help="quick real pass: cap steps/epoch, faithfulness on final epoch only")
    ap.add_argument("--no-amp", dest="no_amp", action="store_true",
                    help="disable fp16 mixed precision")
    a = ap.parse_args()
    a.tiny = a.smoke
    if a.smoke:
        a.epochs, a.out = 1, "/tmp/ravenx_smoke_ckpt"
    train(a)


Writing train_raven_x.py


In [7]:
%%writefile baselines.py
"""RAVEN-X — REAL classification baselines that run on a CPU in seconds.

This is genuine, reportable evidence you can produce today without a GPU:
  B6  TF-IDF (word + char n-grams) + Logistic Regression   (the benchmark's lower bound)

It trains/evaluates on the SAME frozen seed=42 HASOC-2019 Hindi split the transformer
models will use, so the numbers are directly comparable to the MuRIL/XLM-R rows you'll
add from the free-T4 run. No fabrication — actual fit on actual data.

    raven-api/.venv/bin/python raven-codemixed/baselines.py
"""
from __future__ import annotations
import data_loaders as dl
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score
from scipy.sparse import hstack


def vectorize(train_txt, *other_txt):
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=40000)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, max_features=40000)
    Xtr = hstack([word.fit_transform(train_txt), char.fit_transform(train_txt)]).tocsr()
    outs = []
    for txt in other_txt:
        outs.append(hstack([word.transform(txt), char.transform(txt)]).tocsr())
    return Xtr, outs


def report(name, y, pred):
    macro = f1_score(y, pred, average="macro")
    acc = accuracy_score(y, pred)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[1], average=None)
    print(f"  {name:5s}  macro-F1={macro:.4f}  acc={acc:.4f}  "
          f"toxic P={p[0]:.3f} R={r[0]:.3f} F1={f[0]:.3f}")
    return macro


if __name__ == "__main__":
    d = dl.load_hasoc_hindi()
    tr, va, te = d["train"], d["val"], d["test"]
    print(f"HASOC-2019 Hindi frozen split (seed=42): "
          f"train={len(tr)} val={len(va)} test={len(te)}")
    Xtr, (Xva, Xte) = vectorize([x["text"] for x in tr],
                                [x["text"] for x in va], [x["text"] for x in te])
    ytr = [x["label"] for x in tr]; yva = [x["label"] for x in va]; yte = [x["label"] for x in te]

    clf = LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")
    clf.fit(Xtr, ytr)

    print("\nB6 — TF-IDF(word 1-2 + char 2-5) + LogReg  [REAL, CPU]:")
    report("val", yva, clf.predict(Xva))
    test_macro = report("test", yte, clf.predict(Xte))
    print(f"\n>> Honest lower-bound on the frozen test split: macro-F1 = {test_macro:.4f}")
    print(">> The MuRIL/XLM-R rows (from the free-T4 run) go ABOVE this in the same table.")


Writing baselines.py


## 5 · Sanity: audit the data + verify the pipeline (fast)

In [8]:
!python data_loaders.py
!python raven_x.py --smoke

HASOC Hindi: {'train': 3265, 'val': 699, 'test': 701} | train label balance: {0: 1537, 1: 1728}
HateXplain: {'test': 1924, 'train': 15383, 'val': 1922}
  example rationale-bearing post:
   tokens: ['u', 'really', 'think', 'i', 'would', 'not', 'have', 'been', 'raped', 'by', 'feral', 'hindu'] ...
   mask  : [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1] ...  label3= 1
  rationale coverage in train: 9130/15383
[smoke] loading MuRIL tokenizer (real alignment) + tiny random encoder (fast)...
config.json: 100% 411/411 [00:00<00:00, 1.72MB/s]
tokenizer_config.json: 100% 206/206 [00:00<00:00, 837kB/s]
vocab.txt: 100% 3.16M/3.16M [00:00<00:00, 21.8MB/s]
special_tokens_map.json: 100% 113/113 [00:00<00:00, 472kB/s]
[smoke] batch input_ids (8, 64), specials/row≈31, EN rationale tokens=20, ignored=420
[smoke] step 1 gamma=0.0 loss=4.5646 rat_grad_norm=1.6995e+00 | z=0.635 clsA=1.740 clsB=1.455 rat=1.161 comp=0.000 suff=0.000 sparse=0.507 cont=0.048
[smoke] step 2 gamma=0.5 loss=4.5011 rat_grad_norm=1.4175e+0

## 6 · REAL baseline — TF-IDF + LogReg (the benchmark lower bound)

In [9]:
!python baselines.py

HASOC-2019 Hindi frozen split (seed=42): train=3265 val=699 test=701

B6 — TF-IDF(word 1-2 + char 2-5) + LogReg  [REAL, CPU]:
  val    macro-F1=0.7867  acc=0.7868  toxic P=0.818 R=0.768 F1=0.792
  test   macro-F1=0.8130  acc=0.8131  toxic P=0.843 R=0.795 F1=0.818

>> Honest lower-bound on the frozen test split: macro-F1 = 0.8130
>> The MuRIL/XLM-R rows (from the free-T4 run) go ABOVE this in the same table.


## 7 · REAL training — fine-tune MuRIL (the actual model, on the T4)
**Fast mode (~25–45 min):** fp16 + capped steps/epoch + faithfulness on the final epoch. Prints HASOC macro-F1 + HateXplain token-F1 each epoch and **saves a checkpoint every epoch (disconnect-safe).**

For the full paper-grade run later, drop `--fast` and use `--epochs 4` (~1.5 h).

In [10]:
!python train_raven_x.py --model google/muril-base-cased --fast --epochs 3 --out ckpt_ravenx
# full run later:  !python train_raven_x.py --model google/muril-base-cased --epochs 4 --out ckpt_ravenx

/content/train_raven_x.py:131: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
pytorch_model.bin: 100% 953M/953M [00:12<00:00, 74.3MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 27595.02it/s]
[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UN

## 8 · Save your results
Download the checkpoint + zip it from the file browser, or push to Google Drive:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp -r ckpt_ravenx /content/drive/MyDrive/
```
Screenshot the epoch macro-F1 / token-F1 lines — **those are your real, defensible numbers** for the report and viva.